### 04_model_training - 15-06-2026

In [1]:
# 04_model_training.py
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score, classification_report
import gc

print("Loading pre-processed parquet partitions...")
train_df = pd.read_parquet('/Users/abannee/Documents/GitHub/fraud_detection_ml/data/processed/train_engineered.parquet')
val_df = pd.read_parquet('/Users/abannee/Documents/GitHub/fraud_detection_ml/data/processed/val_engineered.parquet')

# Isolate feature vectors from targets cleanly
drop_cols = ['isFraud', 'TransactionID', 'TransactionDT', 'card_proxy_id']
feature_cols = [c for c in train_df.columns if c not in drop_cols]

# Isolate object/categorical columns to prevent runtime model errors
cat_cols = train_df[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Filtering out structural high-cardinality categorical variables for core matrix arrays: {cat_cols}")
final_features = [c for c in feature_cols if c not in cat_cols]

X_train, y_train = train_df[final_features], train_df['isFraud']
X_val, y_val = val_df[final_features], val_df['isFraud']

del train_df, val_df
gc.collect()

Loading pre-processed parquet partitions...
Filtering out structural high-cardinality categorical variables for core matrix arrays: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_75694/2424115758.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train_df[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()


20

In [2]:
print("\n--- Computing Anti-Imbalance Class Scale Weightings ---")
# scale_pos_weight = Total Negative Records / Total Positive Records
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
calculated_scale_weight = neg_count / pos_count
print(f"Negative Controls: {neg_count} | Positive Traps: {pos_count} | Balanced Scale Weight: {calculated_scale_weight:.3f}")

print("\n--- Initializing XGBoost Classifier Architecture ---")
# Configure parameters optimizing on high scale speeds
model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=calculated_scale_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist', # Enable lightning-fast histogram binning
    random_state=42,
    n_jobs=-1
)

print("Fitting gradient boosted decision structures...")
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=10)

print("\n--- Running Validation Set Diagnostics ---")
# Predict risk probability values
val_probs = model.predict_proba(X_val)[:, 1]

# Calculate Precision-Recall Metrics
precision, recall, thresholds = precision_recall_curve(y_val, val_probs)
avg_pr = average_precision_score(y_val, val_probs)
print(f"Validation Average Precision (PR-AUC) Score: {avg_pr:.4f}")

print("\n--- Optimization Pipeline: Finding Optimal Threshold Matrix ---")
# Optimize the F1 score metric dynamically to discover the ideal line balancing risk vs revenue friction
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[best_idx]
print(f"🎯 Optimal Threshold Found: {optimal_threshold:.4f} | Max Achievable Validation F1-Score: {f1_scores[best_idx]:.4f}")

# Generate Precision-Recall Curve Plot
plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label=f'XGBoost Baseline (PR-AUC = {avg_pr:.3f})', color='purple', lw=2)
plt.scatter(recall[best_idx], precision[best_idx], marker='o', color='black', label=f'Optimal Cut-off ({optimal_threshold:.2f})', zorder=5)
plt.title('Validation Precision-Recall Frontier Curve Profile')
plt.xlabel('Recall (Fraud Detection Rate)')
plt.ylabel('Precision (Positive Predictive Rate)')
plt.legend(loc='lower left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('plots/07_precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.close()

# Generate structured class classification printouts using the newly tuned threshold boundary lines
print(f"\n--- Model Classification Metrics at Tuned Threshold Cut-off ({optimal_threshold:.4f}) ---")
tuned_predictions = (val_probs >= optimal_threshold).astype(int)
print(classification_report(y_val, tuned_predictions))

print("Model Compilation and Validation Iteration Complete!")


--- Computing Anti-Imbalance Class Scale Weightings ---
Negative Controls: 398840 | Positive Traps: 14538 | Balanced Scale Weight: 27.434

--- Initializing XGBoost Classifier Architecture ---
Fitting gradient boosted decision structures...
[0]	validation_0-logloss:0.67380
[10]	validation_0-logloss:0.54413
[20]	validation_0-logloss:0.47633
[30]	validation_0-logloss:0.43619
[40]	validation_0-logloss:0.41137
[50]	validation_0-logloss:0.39600
[60]	validation_0-logloss:0.38475
[70]	validation_0-logloss:0.37771
[80]	validation_0-logloss:0.37078
[90]	validation_0-logloss:0.36369
[100]	validation_0-logloss:0.35873
[110]	validation_0-logloss:0.35374
[120]	validation_0-logloss:0.34760
[130]	validation_0-logloss:0.34254
[140]	validation_0-logloss:0.33816
[149]	validation_0-logloss:0.33501

--- Running Validation Set Diagnostics ---
Validation Average Precision (PR-AUC) Score: 0.4863

--- Optimization Pipeline: Finding Optimal Threshold Matrix ---
🎯 Optimal Threshold Found: 0.7830 | Max Achievabl